# Распаковка составного проектора

In [ ]:
import json
import re
import ast

with open("out1.json", encoding="utf-8") as f:
    data = json.load(f)

pattern = re.compile(r"^(\(.+?\)) — (\[.+\])$")

proj_mapping = {}
for line in data["wp_bpe_comparison_str"]:
    m = pattern.match(line)
    key_str, val_str = m.group(1), m.group(2)
    key = ast.literal_eval(key_str)   # tuple (token, id)
    value = ast.literal_eval(val_str) # list[tuple]
    proj_mapping[key] = value

# Распаковка MLP-проектора

In [ ]:
import json
import re
import ast

with open("vocab_mapping_mlp.json", encoding="utf-8") as f:
    data = json.load(f)

pattern = re.compile(r"^(\(.+?\)) — (\[.+\])$")

mlp_mapping = {}
for line in data:
    m = pattern.match(line)
    key_str, val_str = m.group(1), m.group(2)
    key = ast.literal_eval(key_str)
    value = ast.literal_eval(val_str)
    mlp_mapping[key] = value

# Установка моделей

In [1]:
from enum import Enum
from tokenizers import Tokenizer
from torch.utils.data.dataset import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tokenizers.models import BPE, Unigram
from torch.utils.data import DataLoader
import torch

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name_1 = "LiquidAI/LFM2.5-230M"
tokenizer_1 = Tokenizer.from_pretrained(model_name_1)
print(f'Длина словаря {model_name_1}: {tokenizer_1.get_vocab_size()}')
print(f'Модель токенизатора {model_name_1}: {tokenizer_1.model}')

Длина словаря LiquidAI/LFM2.5-230M: 64402
Модель токенизатора LiquidAI/LFM2.5-230M: BPE(dropout=None, unk_token=None, continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"<|pad|>":0, "<|startoftext|>":1, "<|endoftext|>":2, "<|fim_pre|>":3, "<|fim_mid|>":4, ...}, merges=[("Ċ", "Ċ"), ("Ċ", "ĊĊ"), ("ĊĊ", "Ċ"), ("Ċ", "ĊĊĊ"), ("ĊĊ", "ĊĊ"), ...])


In [3]:
model_name_2 = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_2 = Tokenizer.from_pretrained(model_name_2)
print(f'Длина словаря {model_name_2}: {tokenizer_2.get_vocab_size()}')
print(f'Модель токенизатора {model_name_2}: {tokenizer_2.model}')

Длина словаря Qwen/Qwen2.5-0.5B-Instruct: 151665
Модель токенизатора Qwen/Qwen2.5-0.5B-Instruct: BPE(dropout=None, unk_token=None, continuing_subword_prefix="", end_of_word_suffix="", fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"!":0, """:1, "#":2, "$":3, "%":4, ...}, merges=[("Ġ", "Ġ"), ("ĠĠ", "ĠĠ"), ("i", "n"), ("Ġ", "t"), ("ĠĠĠĠ", "ĠĠĠĠ"), ...])


# Установка токенизаторов

In [4]:
tokenizer_model_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 132/132 [00:00<00:00, 924.96it/s]


In [5]:
tokenizer_model_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 927.89it/s] 


In [14]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

class Generation:
    def __init__(self,
                 llm_1: GenerationMixin, 
                 llm_2: GenerationMixin, 
                 tok_1: TokenizersBackend | SentencePieceBackend, 
                 tok_2: TokenizersBackend | SentencePieceBackend,):
        self.llm_1 = llm_1
        self.llm_2 = llm_2
        self.tok_1 = tok_1
        self.tok_2 = tok_2
        self.device = 'cuda'

    def tokenize_one_model(self, 
                           tokenizer_model: TokenizersBackend | SentencePieceBackend, 
                           msg: str,):
        input_ids = tokenizer_model.apply_chat_template(conversation=msg,
                                                        tokenize=True,
                                                        add_generation_prompt=True,
                                                        return_tensors="pt",)["input_ids"].to(self.device)
        return input_ids

    def tokenize_all_models(self, 
                            msg: list, 
                            tokenizers: list,):
        input_ids_list =[]
        for tokenizer in tokenizers:
            input_ids_list.append(self.tokenize_one_model(tokenizer_model=tokenizer, msg=msg))
        return input_ids_list
    
    def generate(self, 
                 llm_inputs: list, 
                 llms: list[GenerationMixin]):
        models_outputs = []
        for llm_input, llm in zip(llm_inputs, llms, strict=True):
            output = llm.generate(llm_input, temperature=0.1, max_new_tokens=512)
            models_outputs.append(output)
        
        return models_outputs

    def decode_generated_ids(self, 
                             output_ids: list, 
                             tokenizers: list[TokenizersBackend]):
        outputs_txt = []
        for output_id, tokenizer in zip(output_ids, tokenizers, strict=True):
            generated_txt = tokenizer.decode(output_id[0], skip_special_tokens=True)
            outputs_txt.append(generated_txt)

        return outputs_txt
        

    def generate_pipe(self, input_str: str):
        msg = [{'role': 'user', 'content': input_str}]
        llm_inputs = self.tokenize_all_models(msg=msg,
                                              tokenizers=[self.tok_1, self.tok_2])
        
        models_outputs_ids = self.generate(llm_inputs=llm_inputs, llms=[self.llm_1, self.llm_2])
        models_outputs_txt = self.decode_generated_ids(output_ids=models_outputs_ids,
                                                       tokenizers=[self.tok_1, self.tok_2])

        print(f'Результаты вывода обоих моделей {models_outputs_txt=}')

In [ ]:
class Aggrement:
    def __init__(self, ):
        pass

    

In [ ]:
agg = Generation(llm_1=model_1, llm_2=model_2, tok_1=tokenizer_model_1, tok_2=tokenizer_model_2)

agg.generate_pipe('Hello booy')

Результаты вывода обоих моделей models_outputs_txt=['user\nHello booy\nassistant\nHello! How can I help you today? 😊', "system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.\nuser\nHello booy\nassistant\nHello! How can I assist you today? If you have any questions or need help with anything specific, feel free to ask. I'm here to help!"]


In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

class GenerationLoop:
    def __init__(self, 
                 model_1: GenerationMixin,
                 model_2: GenerationMixin,
                 tokenizer_1: TokenizersBackend | SentencePieceBackend,
                 tokenizer_2: TokenizersBackend | SentencePieceBackend,
                 mapping_vocab: dict):

In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

class Benchmark:
    def __init__(self,
                 map_proj: dict, 
                 map_mlp: dict, 
                 llm_1: GenerationMixin, 
                 llm_2: GenerationMixin, 
                 tok_1: TokenizersBackend | SentencePieceBackend, 
                 tok_2: TokenizersBackend | SentencePieceBackend,):
        self.map_proj = map_proj
        self.map_mlp = map_mlp
        self.llm_1 = llm_1
        self.llm_2 = llm_2
        self.tok_1 = tok_1
        self.tok_2 = tok_2
    
    def projecting_loop(self, tokens):
        projected_idx_mlp = []
        projected_idx_proj = []
        for idx in tokens:
            token = self.tok_1.decode(idx)
            idx = int(idx)
            print(f'id "{idx}", строка "{token}"')
            if token == '\n': 
                token = 'Ċ'
            if token.startswith(' '):
                token = token.replace(' ',"Ġ")
            
            mlp_projection = self.map_mlp[(token, idx)]
            mtrx_projection = self.map_proj[(token, idx)]

            print(f'НС-отображение {mlp_projection}') 
            print(f'Проекционное отображение {mtrx_projection}') 
            
            projected_idx_mlp.append(mlp_projection[0][1])

            if len(mtrx_projection) > 1:
                for elem in mtrx_projection:
                    print(f'{elem=}')
                    projected_idx_proj.append(elem[1])
            else:
                projected_idx_proj.append(mtrx_projection[0][1])
            print('##########################################')
        

        # print(f'{projected_arr_mlp=}')
        # print(f'{projected_arr_proj=}')
        projected_only_idx_mlp = []
        print(f'Декодированная строка {self.tok_2.decode(projected_idx_mlp)}')
        print(f'Декодированная строка {self.tok_2.decode(projected_idx_proj)}')


    def generation_pipe(self, msg: list):
        text = self.tok_1.apply_chat_template(msg,
                                              tokenize=True,
                                              return_tensors='pt',
                                              add_generation_prompt=False)
        
        tokens_1 = text['input_ids'][0]
        strings = self.tok_1.decode(tokens_1, skip_special_tokens=True)
        return tokens_1, strings

    def map_input(self, input: str):
        message = [{'role': 'user', 'content': input}]

        tokens, strings = self.generation_pipe(msg=message)

        print(f'Разбивка по токенам: {tokens=}')
        print(f'Декодировано в текст: {strings=}')

        self.projecting_loop(tokens)

In [ ]:
bench = Benchmark(map_proj=proj_mapping,
                  map_mlp=mlp_mapping,
                  llm_1=model_1,
                  llm_2=model_2,
                  tok_1=tokenizer_model_1,
                  tok_2=tokenizer_model_2)

bench.map_input(input="The file, 90 megabytes in size, downloads at the rate of 5 megabytes per second for its first 60 megabytes, and then 10 megabytes per second thereafter. How long, in seconds, does it take to download entirely?")